# AuraGateway worker-observability harness recovery materializer v1

This complete replacement notebook handles Kaggle's auto-expanded source ZIP representation.

Use exactly:

- **Kaggle title:** `ag-worker-obs-harness-materializer-v1`
- **Input:** `ag-worker-obs-harness-dceda98-v1-input` Version 1
- **Accelerator:** None
- **Internet:** Off
- **Secrets:** None
- **Execution:** Save Version → Save & Run All once

The notebook validates the current package manifest, source receipt, canonical inventory, and every expanded source file. It reconstructs the exact Windows-produced deterministic ZIP, verifies the frozen archive SHA-256, materializes the 1,076-file harness, and emits a bounded receipt.

It performs no installation, model load, worker startup, request, authorization, manifest promotion, or GPU execution.


In [ ]:
from __future__ import annotations

import hashlib
import json
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath

NOTEBOOK_NAME = "ag-worker-obs-harness-materializer-v1"
DATASET_NAME = "ag-worker-obs-harness-dceda98-v1-input"
INPUT_ROOT = Path("/kaggle/input").resolve()
WORK_ROOT = Path("/kaggle/working").resolve()

EXPECTED_SOURCE_COMMIT = "dceda98989386de7a4d57616f9f8a8023f866f10"
EXPECTED_REVIEW_MERGE_COMMIT = "997efb4aacf998567a3d92e7202a0054bf473ca4"
EXPECTED_SOURCE_SHORT_COMMIT = "dceda98"

EXPECTED_EXPANDED_DIRECTORY = "ag-worker-obs-harness-dceda98-v1"
EXPECTED_ARCHIVE_FILENAME = "ag-worker-obs-harness-dceda98-v1.zip"
EXPECTED_ARCHIVE_SHA256 = "cf6a926a129699da00602a23650dbce5645bc28527ba52373bf9dd55beaccf3f"
EXPECTED_ARCHIVE_SIZE_BYTES = 2765735

EXPECTED_ORIGINAL_MATERIALIZER_FILENAME = "ag_worker_obs_harness_materializer_v1.ipynb"
EXPECTED_ORIGINAL_MATERIALIZER_SHA256 = "2ad7cd527ccd11efda73a86c96c73be5518271982aa72ecd832ac1e0302e6133"
EXPECTED_ORIGINAL_MATERIALIZER_SIZE_BYTES = 11653

EXPECTED_PACKAGE_MANIFEST_FILENAME = "package_manifest.json"
EXPECTED_PACKAGE_MANIFEST_SHA256 = "19eb8ea9174b53cf77c14c7688bab6f971282b50a55458b6a056e5e6240daad4"
EXPECTED_PACKAGE_MANIFEST_SIZE_BYTES = 805

EXPECTED_SOURCE_INVENTORY_FILENAME = "source_inventory.json"
EXPECTED_SOURCE_INVENTORY_SHA256 = "c66f2589bdf55ab34f82bffc1eaaa4b4c7e73cb8195867333ccd99a58438f3e4"
EXPECTED_SOURCE_INVENTORY_SIZE_BYTES = 181483

EXPECTED_SOURCE_RECEIPT_FILENAME = "source_receipt.json"
EXPECTED_SOURCE_RECEIPT_SHA256 = "a5241d9dda8f3a71a6e9923f12c4194168e8860bf2f6d07e15f4f90342a73662"
EXPECTED_SOURCE_RECEIPT_SIZE_BYTES = 1300

EXPECTED_DIRECTORY_SHA256 = "c66f2589bdf55ab34f82bffc1eaaa4b4c7e73cb8195867333ccd99a58438f3e4"
EXPECTED_FILE_COUNT = 1076
EXPECTED_TOTAL_BYTES = 10850278
EXPECTED_OUTPUT_DIRECTORY = "auragateway_qualification_harness_dceda98_worker_obs_v1"

OUTPUT_PARENT = WORK_ROOT / "ag_worker_obs_harness_materializer_v1_output"
OUTPUT_ROOT = OUTPUT_PARENT / EXPECTED_OUTPUT_DIRECTORY
MATERIALIZATION_RECEIPT_PATH = OUTPUT_PARENT / "materialization_receipt.json"

STAGING_ROOT = WORK_ROOT / ".ag_worker_obs_harness_materializer_v1_staging"
STAGING_ARCHIVE_PATH = STAGING_ROOT / EXPECTED_ARCHIVE_FILENAME
STAGING_HARNESS_ROOT = STAGING_ROOT / EXPECTED_OUTPUT_DIRECTORY

FAILURE_ROOT = WORK_ROOT / "ag_worker_obs_harness_materializer_v1_failure"
FAILURE_ZIP_PATH = WORK_ROOT / "ag-worker-obs-harness-materializer-v1-failure.zip"

MAXIMUM_FILES = 5_000
MAXIMUM_TOTAL_BYTES = 100 * 1024 * 1024
ZIP_TIMESTAMP = (1980, 1, 1, 0, 0, 0)
ARCHIVE_SUFFIXES = (
    ".zip",
    ".tar",
    ".tgz",
    ".gz",
    ".bz2",
    ".xz",
    ".7z",
    ".whl",
)
REQUIRED_SOURCE_PATHS = {
    "pyproject.toml",
    "src/auragateway/local_abc/full_abc_local_environment_qualification_kaggle_runtime_adapter.py",
    "src/auragateway/local_abc/full_abc_local_environment_qualification_kaggle_launcher.py",
    "src/auragateway/local_abc/full_abc_local_environment_qualification_worker_startup_diagnostics.py",
    "notebooks/auragateway_full_abc_environment_qualification_launcher_v1.ipynb",
}


def canonical_json(payload: object) -> str:
    return json.dumps(
        payload,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


def bytes_sha256(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def normalized_relative_path(value: str) -> PurePosixPath:
    path = PurePosixPath(value)
    if (
        path.is_absolute()
        or not path.parts
        or ".." in path.parts
        or "\\" in value
        or value.startswith("./")
        or path.as_posix() != value
    ):
        raise RuntimeError("source inventory contains an unsafe relative path")
    if value.lower().endswith(ARCHIVE_SUFFIXES):
        raise RuntimeError("source inventory contains a nested archive")
    return path


def load_canonical_json(path: Path) -> tuple[object, bytes]:
    raw = path.read_bytes()
    try:
        payload = json.loads(raw.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as exc:
        raise RuntimeError(f"invalid canonical JSON control: {path.name}") from exc

    if raw != canonical_json(payload).encode("utf-8"):
        raise RuntimeError(f"JSON control is not canonical: {path.name}")

    return payload, raw


def validate_regular_file(
    path: Path,
    *,
    expected_sha256: str,
    expected_size_bytes: int,
) -> None:
    if path.is_symlink() or not path.is_file():
        raise RuntimeError(f"required source-package control is missing or non-regular: {path.name}")

    metadata = path.stat()
    if not stat.S_ISREG(metadata.st_mode):
        raise RuntimeError(f"source-package control is non-regular: {path.name}")
    if metadata.st_size != expected_size_bytes:
        raise RuntimeError(f"source-package control size drifted: {path.name}")
    if file_sha256(path) != expected_sha256:
        raise RuntimeError(f"source-package control SHA-256 drifted: {path.name}")


def resolve_dataset_root() -> Path:
    candidates = tuple(
        path.resolve()
        for path in INPUT_ROOT.rglob(DATASET_NAME)
        if path.is_dir() and not path.is_symlink()
    )
    if len(candidates) != 1:
        raise RuntimeError(
            "expected exactly one attached worker-observability source dataset "
            f"but observed {len(candidates)}"
        )

    dataset_root = candidates[0]
    if INPUT_ROOT != dataset_root and INPUT_ROOT not in dataset_root.parents:
        raise RuntimeError("worker-observability source dataset escaped /kaggle/input")
    return dataset_root


def validate_dataset_topology(dataset_root: Path) -> Path:
    expected_names = {
        EXPECTED_EXPANDED_DIRECTORY,
        EXPECTED_ORIGINAL_MATERIALIZER_FILENAME,
        EXPECTED_PACKAGE_MANIFEST_FILENAME,
        EXPECTED_SOURCE_INVENTORY_FILENAME,
        EXPECTED_SOURCE_RECEIPT_FILENAME,
    }

    observed_names: set[str] = set()
    for path in dataset_root.iterdir():
        if path.is_symlink():
            raise RuntimeError("source dataset contains a symbolic link")
        observed_names.add(path.name)

    if observed_names != expected_names:
        raise RuntimeError(
            "expanded source dataset top-level set drifted: "
            f"expected={sorted(expected_names)} observed={sorted(observed_names)}"
        )

    expanded_root = dataset_root / EXPECTED_EXPANDED_DIRECTORY
    if expanded_root.is_symlink() or not expanded_root.is_dir():
        raise RuntimeError("expected Kaggle-expanded source directory is missing or symbolic")

    validate_regular_file(
        dataset_root / EXPECTED_ORIGINAL_MATERIALIZER_FILENAME,
        expected_sha256=EXPECTED_ORIGINAL_MATERIALIZER_SHA256,
        expected_size_bytes=EXPECTED_ORIGINAL_MATERIALIZER_SIZE_BYTES,
    )
    validate_regular_file(
        dataset_root / EXPECTED_PACKAGE_MANIFEST_FILENAME,
        expected_sha256=EXPECTED_PACKAGE_MANIFEST_SHA256,
        expected_size_bytes=EXPECTED_PACKAGE_MANIFEST_SIZE_BYTES,
    )
    validate_regular_file(
        dataset_root / EXPECTED_SOURCE_INVENTORY_FILENAME,
        expected_sha256=EXPECTED_SOURCE_INVENTORY_SHA256,
        expected_size_bytes=EXPECTED_SOURCE_INVENTORY_SIZE_BYTES,
    )
    validate_regular_file(
        dataset_root / EXPECTED_SOURCE_RECEIPT_FILENAME,
        expected_sha256=EXPECTED_SOURCE_RECEIPT_SHA256,
        expected_size_bytes=EXPECTED_SOURCE_RECEIPT_SIZE_BYTES,
    )
    return expanded_root


def load_and_validate_controls(
    dataset_root: Path,
) -> tuple[dict[str, object], dict[str, object], dict[str, object]]:
    manifest_payload, _ = load_canonical_json(
        dataset_root / EXPECTED_PACKAGE_MANIFEST_FILENAME
    )
    inventory_payload, inventory_raw = load_canonical_json(
        dataset_root / EXPECTED_SOURCE_INVENTORY_FILENAME
    )
    receipt_payload, _ = load_canonical_json(
        dataset_root / EXPECTED_SOURCE_RECEIPT_FILENAME
    )

    if not isinstance(manifest_payload, dict):
        raise RuntimeError("package manifest root must be one object")
    if not isinstance(inventory_payload, dict):
        raise RuntimeError("source inventory root must be one object")
    if not isinstance(receipt_payload, dict):
        raise RuntimeError("source receipt root must be one object")

    manifest = dict(manifest_payload)
    inventory = dict(inventory_payload)
    receipt = dict(receipt_payload)

    expected_manifest_keys = {
        "schema_version",
        "package_id",
        "source_commit",
        "files",
        "authorization_included",
        "kaggle_execution_performed",
        "model_requests_performed",
    }
    if set(manifest) != expected_manifest_keys:
        raise RuntimeError("package manifest key set drifted")

    expected_manifest_values = {
        "schema_version": "1.0.0",
        "package_id": "auragateway-worker-observability-harness-source-v1",
        "source_commit": EXPECTED_SOURCE_COMMIT,
        "authorization_included": False,
        "kaggle_execution_performed": False,
        "model_requests_performed": 0,
    }
    for key, expected_value in expected_manifest_values.items():
        if manifest.get(key) != expected_value:
            raise RuntimeError(f"package manifest field drifted: {key}")

    raw_manifest_files = manifest.get("files")
    if not isinstance(raw_manifest_files, list):
        raise RuntimeError("package manifest files must be one array")

    manifest_files: dict[str, dict[str, object]] = {}
    for raw_entry in raw_manifest_files:
        if not isinstance(raw_entry, dict):
            raise RuntimeError("package manifest contains an invalid file entry")
        if set(raw_entry) != {"path", "sha256", "size_bytes"}:
            raise RuntimeError("package manifest file-entry key set drifted")

        path_value = raw_entry.get("path")
        sha256_value = raw_entry.get("sha256")
        size_value = raw_entry.get("size_bytes")
        if not isinstance(path_value, str):
            raise RuntimeError("package manifest file path is invalid")
        if PurePosixPath(path_value).name != path_value:
            raise RuntimeError("package manifest file path is not a safe top-level name")
        if path_value in manifest_files:
            raise RuntimeError("package manifest contains duplicate file paths")
        if (
            not isinstance(sha256_value, str)
            or len(sha256_value) != 64
            or any(character not in "0123456789abcdef" for character in sha256_value)
        ):
            raise RuntimeError("package manifest file SHA-256 is invalid")
        if (
            not isinstance(size_value, int)
            or isinstance(size_value, bool)
            or size_value < 0
        ):
            raise RuntimeError("package manifest file size is invalid")
        manifest_files[path_value] = dict(raw_entry)

    expected_manifest_files = {
        EXPECTED_ARCHIVE_FILENAME: {
            "sha256": EXPECTED_ARCHIVE_SHA256,
            "size_bytes": EXPECTED_ARCHIVE_SIZE_BYTES,
        },
        EXPECTED_ORIGINAL_MATERIALIZER_FILENAME: {
            "sha256": EXPECTED_ORIGINAL_MATERIALIZER_SHA256,
            "size_bytes": EXPECTED_ORIGINAL_MATERIALIZER_SIZE_BYTES,
        },
        EXPECTED_SOURCE_INVENTORY_FILENAME: {
            "sha256": EXPECTED_SOURCE_INVENTORY_SHA256,
            "size_bytes": EXPECTED_SOURCE_INVENTORY_SIZE_BYTES,
        },
        EXPECTED_SOURCE_RECEIPT_FILENAME: {
            "sha256": EXPECTED_SOURCE_RECEIPT_SHA256,
            "size_bytes": EXPECTED_SOURCE_RECEIPT_SIZE_BYTES,
        },
    }
    if set(manifest_files) != set(expected_manifest_files):
        raise RuntimeError("package manifest file set drifted")

    for filename, expected_identity in expected_manifest_files.items():
        entry = manifest_files[filename]
        if entry.get("sha256") != expected_identity["sha256"]:
            raise RuntimeError(f"package manifest SHA-256 drifted: {filename}")
        if entry.get("size_bytes") != expected_identity["size_bytes"]:
            raise RuntimeError(f"package manifest size drifted: {filename}")

    expected_receipt_keys = {
        "active_manifest_promoted",
        "archive_filename",
        "archive_sha256",
        "authorization_issued",
        "benchmark_trajectory_requests_performed",
        "gpu_execution_performed",
        "input_dataset_name",
        "materializer_notebook_filename",
        "materializer_notebook_name",
        "model_loaded",
        "model_requests_performed",
        "nested_archives_present",
        "network_access_performed",
        "output_directory",
        "package_installation_performed",
        "review_merge_commit",
        "schema_version",
        "source_commit",
        "source_directory_sha256",
        "source_file_count",
        "source_inventory_filename",
        "source_inventory_sha256",
        "source_short_commit",
        "source_total_bytes",
        "status",
        "symlinks_present",
        "worker_started",
    }
    if set(receipt) != expected_receipt_keys:
        raise RuntimeError("source receipt key set drifted")

    expected_receipt_values = {
        "active_manifest_promoted": False,
        "archive_filename": EXPECTED_ARCHIVE_FILENAME,
        "archive_sha256": EXPECTED_ARCHIVE_SHA256,
        "authorization_issued": False,
        "benchmark_trajectory_requests_performed": 0,
        "gpu_execution_performed": False,
        "input_dataset_name": DATASET_NAME,
        "materializer_notebook_filename": EXPECTED_ORIGINAL_MATERIALIZER_FILENAME,
        "materializer_notebook_name": NOTEBOOK_NAME,
        "model_loaded": False,
        "model_requests_performed": 0,
        "nested_archives_present": False,
        "network_access_performed": False,
        "output_directory": EXPECTED_OUTPUT_DIRECTORY,
        "package_installation_performed": False,
        "review_merge_commit": EXPECTED_REVIEW_MERGE_COMMIT,
        "schema_version": "1.0.0",
        "source_commit": EXPECTED_SOURCE_COMMIT,
        "source_directory_sha256": EXPECTED_DIRECTORY_SHA256,
        "source_file_count": EXPECTED_FILE_COUNT,
        "source_inventory_filename": EXPECTED_SOURCE_INVENTORY_FILENAME,
        "source_inventory_sha256": EXPECTED_SOURCE_INVENTORY_SHA256,
        "source_short_commit": EXPECTED_SOURCE_SHORT_COMMIT,
        "source_total_bytes": EXPECTED_TOTAL_BYTES,
        "status": "WORKER_OBSERVABILITY_HARNESS_SOURCE_PACKAGED",
        "symlinks_present": False,
        "worker_started": False,
    }
    for key, expected_value in expected_receipt_values.items():
        if receipt.get(key) != expected_value:
            raise RuntimeError(f"source receipt field drifted: {key}")

    if bytes_sha256(inventory_raw) != EXPECTED_SOURCE_INVENTORY_SHA256:
        raise RuntimeError("source inventory bytes drifted")
    if receipt["source_inventory_sha256"] != bytes_sha256(inventory_raw):
        raise RuntimeError("source receipt inventory identity drifted")
    if receipt["source_directory_sha256"] != bytes_sha256(inventory_raw):
        raise RuntimeError("source directory identity drifted from canonical inventory")

    return manifest, inventory, receipt


def validate_inventory(
    inventory: dict[str, object],
) -> tuple[list[dict[str, object]], dict[str, dict[str, object]]]:
    if set(inventory) != {"schema_version", "files"}:
        raise RuntimeError("source inventory root key set drifted")
    if inventory.get("schema_version") != "1.0.0":
        raise RuntimeError("source inventory schema version drifted")

    raw_files = inventory.get("files")
    if not isinstance(raw_files, list):
        raise RuntimeError("source inventory files must be one array")
    if not raw_files or len(raw_files) > MAXIMUM_FILES:
        raise RuntimeError("source inventory file count is outside the approved boundary")

    entries: list[dict[str, object]] = []
    by_path: dict[str, dict[str, object]] = {}
    ordered_paths: list[str] = []
    total_bytes = 0

    for raw_entry in raw_files:
        if not isinstance(raw_entry, dict):
            raise RuntimeError("source inventory contains an invalid entry")
        if set(raw_entry) != {"path", "sha256", "size_bytes"}:
            raise RuntimeError("source inventory entry key set drifted")

        path_value = raw_entry.get("path")
        sha256_value = raw_entry.get("sha256")
        size_value = raw_entry.get("size_bytes")

        if not isinstance(path_value, str):
            raise RuntimeError("source inventory path is invalid")
        relative_path = normalized_relative_path(path_value).as_posix()
        if relative_path in by_path:
            raise RuntimeError("source inventory contains duplicate paths")
        if (
            not isinstance(sha256_value, str)
            or len(sha256_value) != 64
            or any(character not in "0123456789abcdef" for character in sha256_value)
        ):
            raise RuntimeError("source inventory SHA-256 is invalid")
        if (
            not isinstance(size_value, int)
            or isinstance(size_value, bool)
            or size_value < 0
        ):
            raise RuntimeError("source inventory size is invalid")

        entry = {
            "path": relative_path,
            "sha256": sha256_value,
            "size_bytes": size_value,
        }
        entries.append(entry)
        by_path[relative_path] = entry
        ordered_paths.append(relative_path)
        total_bytes += size_value

        if total_bytes > MAXIMUM_TOTAL_BYTES:
            raise RuntimeError("source inventory exceeds the approved byte boundary")

    if ordered_paths != sorted(ordered_paths):
        raise RuntimeError("source inventory path order is not canonical")
    if len(entries) != EXPECTED_FILE_COUNT:
        raise RuntimeError("source inventory file count drifted")
    if total_bytes != EXPECTED_TOTAL_BYTES:
        raise RuntimeError("source inventory total bytes drifted")
    if not REQUIRED_SOURCE_PATHS.issubset(by_path):
        missing = sorted(REQUIRED_SOURCE_PATHS.difference(by_path))
        raise RuntimeError(f"source inventory is missing required paths: {missing}")

    canonical_inventory = canonical_json(
        {"schema_version": "1.0.0", "files": entries}
    ).encode("utf-8")
    if bytes_sha256(canonical_inventory) != EXPECTED_DIRECTORY_SHA256:
        raise RuntimeError("source inventory canonical identity drifted")

    return entries, by_path


def inspect_expanded_tree(
    expanded_root: Path,
    by_path: dict[str, dict[str, object]],
) -> tuple[list[dict[str, object]], int]:
    observed_entries: list[dict[str, object]] = []
    observed_paths: list[str] = []
    total_bytes = 0

    for path in sorted(expanded_root.rglob("*"), key=lambda item: item.as_posix()):
        if path.is_symlink():
            raise RuntimeError("expanded source tree contains a symbolic link")
        if path.is_dir():
            continue

        metadata = path.stat()
        if not stat.S_ISREG(metadata.st_mode):
            raise RuntimeError("expanded source tree contains a non-regular member")

        relative_path = path.relative_to(expanded_root).as_posix()
        normalized_relative_path(relative_path)
        expected = by_path.get(relative_path)
        if expected is None:
            raise RuntimeError(
                f"expanded source tree contains an unexpected file: {relative_path}"
            )
        if metadata.st_size != expected["size_bytes"]:
            raise RuntimeError(f"expanded source file size drifted: {relative_path}")
        if file_sha256(path) != expected["sha256"]:
            raise RuntimeError(f"expanded source file SHA-256 drifted: {relative_path}")

        observed_entry = {
            "path": relative_path,
            "sha256": expected["sha256"],
            "size_bytes": metadata.st_size,
        }
        observed_entries.append(observed_entry)
        observed_paths.append(relative_path)
        total_bytes += metadata.st_size

        if len(observed_entries) > MAXIMUM_FILES:
            raise RuntimeError("expanded source tree exceeds the file-count boundary")
        if total_bytes > MAXIMUM_TOTAL_BYTES:
            raise RuntimeError("expanded source tree exceeds the byte boundary")

    expected_paths = sorted(by_path)
    if observed_paths != expected_paths:
        missing = sorted(set(expected_paths).difference(observed_paths))
        extra = sorted(set(observed_paths).difference(expected_paths))
        raise RuntimeError(
            f"expanded source tree path set drifted: missing={missing} extra={extra}"
        )
    if len(observed_entries) != EXPECTED_FILE_COUNT:
        raise RuntimeError("expanded source tree file count drifted")
    if total_bytes != EXPECTED_TOTAL_BYTES:
        raise RuntimeError("expanded source tree total bytes drifted")

    observed_inventory = canonical_json(
        {"schema_version": "1.0.0", "files": observed_entries}
    ).encode("utf-8")
    if bytes_sha256(observed_inventory) != EXPECTED_DIRECTORY_SHA256:
        raise RuntimeError("expanded source tree canonical identity drifted")

    return observed_entries, total_bytes


def reconstruct_exact_archive(
    expanded_root: Path,
    entries: list[dict[str, object]],
) -> None:
    if STAGING_ARCHIVE_PATH.exists():
        raise RuntimeError("staging archive already exists")

    with zipfile.ZipFile(
        STAGING_ARCHIVE_PATH,
        mode="x",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as archive:
        for entry in entries:
            relative_path = str(entry["path"])
            source_path = expanded_root.joinpath(
                *normalized_relative_path(relative_path).parts
            )

            info = zipfile.ZipInfo(relative_path, date_time=ZIP_TIMESTAMP)
            # The source archive was built by Python on Windows. ZipInfo defaults
            # to create_system=0 there. Set it explicitly so Kaggle/Linux produces
            # byte-identical central-directory metadata.
            info.create_system = 0
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o100644 << 16
            archive.writestr(info, source_path.read_bytes())

    if STAGING_ARCHIVE_PATH.stat().st_size != EXPECTED_ARCHIVE_SIZE_BYTES:
        raise RuntimeError("reconstructed source archive size drifted")
    if file_sha256(STAGING_ARCHIVE_PATH) != EXPECTED_ARCHIVE_SHA256:
        raise RuntimeError("reconstructed source archive SHA-256 drifted")

    with zipfile.ZipFile(STAGING_ARCHIVE_PATH) as archive:
        members = archive.infolist()
        observed_names = [member.filename for member in members]
        expected_names = [str(entry["path"]) for entry in entries]
        if observed_names != expected_names:
            raise RuntimeError("reconstructed source archive member order drifted")
        if any(member.is_dir() for member in members):
            raise RuntimeError("reconstructed source archive contains directory members")
        if any(member.flag_bits & 0x1 for member in members):
            raise RuntimeError("reconstructed source archive contains encrypted members")


def copy_materialized_tree(
    expanded_root: Path,
    entries: list[dict[str, object]],
) -> None:
    STAGING_HARNESS_ROOT.mkdir()

    for entry in entries:
        relative_path = str(entry["path"])
        source = expanded_root.joinpath(
            *normalized_relative_path(relative_path).parts
        )
        destination = STAGING_HARNESS_ROOT.joinpath(
            *normalized_relative_path(relative_path).parts
        )
        destination.parent.mkdir(parents=True, exist_ok=True)

        with source.open("rb") as source_handle, destination.open("xb") as target:
            shutil.copyfileobj(source_handle, target, length=1024 * 1024)


def write_failure_artifact(exc: Exception) -> None:
    shutil.rmtree(FAILURE_ROOT, ignore_errors=True)
    FAILURE_ZIP_PATH.unlink(missing_ok=True)
    FAILURE_ROOT.mkdir()

    failure_payload = {
        "schema_version": "1.0.0",
        "status": "WORKER_OBSERVABILITY_HARNESS_MATERIALIZATION_FAILED",
        "source_commit": EXPECTED_SOURCE_COMMIT,
        "producer_notebook_name": NOTEBOOK_NAME,
        "error_type": type(exc).__name__,
        "safe_message": str(exc)[:500],
        "gpu_execution_performed": False,
        "network_access_performed": False,
        "package_installation_performed": False,
        "model_loaded": False,
        "worker_started": False,
        "model_requests_performed": 0,
        "authorization_issued": False,
    }
    failure_path = FAILURE_ROOT / "failure.json"
    failure_path.write_text(canonical_json(failure_payload), encoding="utf-8")

    manifest_payload = {
        "schema_version": "1.0.0",
        "files": [
            {
                "path": "failure.json",
                "sha256": file_sha256(failure_path),
                "size_bytes": failure_path.stat().st_size,
            }
        ],
    }
    manifest_path = FAILURE_ROOT / "failure_manifest.json"
    manifest_path.write_text(canonical_json(manifest_payload), encoding="utf-8")

    with zipfile.ZipFile(
        FAILURE_ZIP_PATH,
        mode="x",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as archive:
        for path in (failure_path, manifest_path):
            info = zipfile.ZipInfo(path.name, date_time=ZIP_TIMESTAMP)
            info.create_system = 3
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o100644 << 16
            archive.writestr(info, path.read_bytes())

    print("status=WORKER_OBSERVABILITY_HARNESS_MATERIALIZATION_FAILED")
    print(f"error_type={type(exc).__name__}")
    print(f"safe_message={str(exc)[:500]}")
    print(f"failure_artifact={FAILURE_ZIP_PATH}")


def run_materialization() -> None:
    if len(NOTEBOOK_NAME) > 50:
        raise RuntimeError("Kaggle notebook name exceeds 50 characters")
    if OUTPUT_PARENT.exists():
        raise RuntimeError("worker-observability materializer output already exists")
    if STAGING_ROOT.exists():
        raise RuntimeError("worker-observability materializer staging already exists")
    if FAILURE_ROOT.exists() or FAILURE_ZIP_PATH.exists():
        raise RuntimeError("worker-observability materializer failure output already exists")

    dataset_root = resolve_dataset_root()
    expanded_root = validate_dataset_topology(dataset_root)
    _, inventory, _ = load_and_validate_controls(dataset_root)
    entries, by_path = validate_inventory(inventory)
    inspect_expanded_tree(expanded_root, by_path)

    STAGING_ROOT.mkdir()
    reconstruct_exact_archive(expanded_root, entries)
    copy_materialized_tree(expanded_root, entries)
    materialized_entries, materialized_total_bytes = inspect_expanded_tree(
        STAGING_HARNESS_ROOT,
        by_path,
    )

    OUTPUT_PARENT.mkdir()
    STAGING_HARNESS_ROOT.replace(OUTPUT_ROOT)

    materialization_receipt = {
        "schema_version": "1.0.0",
        "status": "WORKER_OBSERVABILITY_HARNESS_MATERIALIZED",
        "producer_notebook_name": NOTEBOOK_NAME,
        "source_commit": EXPECTED_SOURCE_COMMIT,
        "input_dataset_name": DATASET_NAME,
        "input_mode": "kaggle_expanded_archive_recovery",
        "archive_filename": EXPECTED_ARCHIVE_FILENAME,
        "archive_sha256": EXPECTED_ARCHIVE_SHA256,
        "archive_reconstructed": True,
        "source_inventory_sha256": EXPECTED_SOURCE_INVENTORY_SHA256,
        "source_package_manifest_sha256": EXPECTED_PACKAGE_MANIFEST_SHA256,
        "source_receipt_sha256": EXPECTED_SOURCE_RECEIPT_SHA256,
        "original_materializer_notebook_sha256": EXPECTED_ORIGINAL_MATERIALIZER_SHA256,
        "output_directory": EXPECTED_OUTPUT_DIRECTORY,
        "directory_sha256": EXPECTED_DIRECTORY_SHA256,
        "file_count": len(materialized_entries),
        "total_bytes": materialized_total_bytes,
        "nested_archives_present": False,
        "symlinks_present": False,
        "network_access_performed": False,
        "package_installation_performed": False,
        "gpu_execution_performed": False,
        "model_loaded": False,
        "worker_started": False,
        "model_requests_performed": 0,
        "benchmark_trajectory_requests_performed": 0,
        "authorization_issued": False,
        "active_manifest_promoted": False,
    }
    MATERIALIZATION_RECEIPT_PATH.write_text(
        canonical_json(materialization_receipt),
        encoding="utf-8",
    )

    STAGING_ARCHIVE_PATH.unlink()
    STAGING_ROOT.rmdir()

    print("status=WORKER_OBSERVABILITY_HARNESS_MATERIALIZED")
    print(f"source_commit={EXPECTED_SOURCE_COMMIT}")
    print("input_mode=kaggle_expanded_archive_recovery")
    print("archive_reconstructed=true")
    print(f"archive_sha256={EXPECTED_ARCHIVE_SHA256}")
    print(f"output_directory={EXPECTED_OUTPUT_DIRECTORY}")
    print(f"output_path={OUTPUT_ROOT}")
    print(f"file_count={len(materialized_entries)}")
    print(f"total_bytes={materialized_total_bytes}")
    print(f"directory_sha256={EXPECTED_DIRECTORY_SHA256}")
    print("nested_archives_present=false")
    print("gpu_execution_performed=false")
    print("network_access_performed=false")
    print("package_installation_performed=false")
    print("authorization_issued=false")
    print("model_requests_performed=0")
    print("active_manifest_promoted=false")
    print("save_this_notebook_output=true")
    print("next_gate=metadata_only_operational_input_inspection")


try:
    run_materialization()
except Exception as exc:
    shutil.rmtree(OUTPUT_PARENT, ignore_errors=True)
    shutil.rmtree(STAGING_ROOT, ignore_errors=True)
    try:
        write_failure_artifact(exc)
    except Exception as failure_exc:
        print("failure_artifact_write_failed=true")
        print(f"failure_artifact_error_type={type(failure_exc).__name__}")
    raise
